# Class evaluation

In this notebook we are trying to match all the keyword groups into documents to highlights specific patterns

## Load libraries

In [1]:
import os
import sys

project_root = os.path.abspath(os.path.join(os.getcwd(), "../.."))

if project_root not in sys.path:
    sys.path.append(project_root)

In [2]:
import pipeline.src.python.config as cfg
import pandas as pd
import numpy as np
pd.set_option('display.max_colwidth', None)
from bertopic import BERTopic
import polars as pl
import re

/home/banfi/.uve/cuda/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Load data

In [ ]:
cluster_df = pd.read_parquet('community_detection_results_4B_0_0_5.parquet')

In [42]:
model_list = ['scopus','science_news','the_guardian']

In [43]:
class_eval_df = pd.DataFrame()

In [44]:
for model in model_list:
    
    cfg_dict = cfg.MAGAZINE_CONFIG[model]
    bertopic_model = BERTopic.load(cfg_dict['REFERENCE_MODEL'])

    model_data = np.load(cfg_dict['OUTPUT_PATH'],allow_pickle=True)

    dataset = pd.read_parquet(cfg_dict['DATASET_PATH'])
    
    tmp_df = bertopic_model.get_document_info(model_data['text'])['CustomName'].reset_index()
    tmp_df = tmp_df.drop(columns='index')

    tmp_df['id'] =  list(model_data['id'])
    tmp_df = tmp_df.rename(columns={'CustomName':'Topic Label'})
    
    tmp_df = tmp_df.merge(cluster_df,on='Topic Label')

    columns_to_mantain = ['id','text']
    columns_to_drop = [column for column in dataset.columns if column not in columns_to_mantain ]

    tmp_df = tmp_df.merge(dataset.drop(columns=columns_to_drop),on='id')

    class_eval_df = pd.concat([class_eval_df,tmp_df])


2026-05-27 17:00:45,254 - BERTopic - WARNING: You are loading a BERTopic model without explicitly defining an embedding model. If you want to also load in an embedding model, make sure to use `BERTopic.load(my_model, embedding_model=my_embedding_model)`.
2026-05-27 17:00:59,352 - BERTopic - WARNING: You are loading a BERTopic model without explicitly defining an embedding model. If you want to also load in an embedding model, make sure to use `BERTopic.load(my_model, embedding_model=my_embedding_model)`.
2026-05-27 17:01:00,381 - BERTopic - WARNING: You are loading a BERTopic model without explicitly defining an embedding model. If you want to also load in an embedding model, make sure to use `BERTopic.load(my_model, embedding_model=my_embedding_model)`.


In [45]:
class_eval_df_polars = pl.from_pandas(class_eval_df)

In [46]:
n_clusters = class_eval_df_polars['Cluster'].max()

## Search keywords into datasets

In [47]:
from pipeline.src.python.data.interim.vocabulary.vocabulary import (
    BACTERIAL_PATHOGENS,
    SEQUENCING_METHODS,
    METHODS,
    GENOME_BASED_ANALYSES,
    FIELD,
    EVOLUTIONARY_MECHANISMS,
    TRANSMISSION_EPIDEMIOLOGY,  
    SURVEILLANCE_AND_PUBLIC_HEALTH, 
    PLATFORM_AND_TECHNOLOGIES,  
    BIOINFORMATIC_TOOLS,    
    CONSORTIA_AND_DATABASES,    
    VARIANT_AND_GENETIC_MARKER, 
    IMMUNOLOGY_TOOLS_AND_CONCEPTS, 
    HAI,    
    AMR,    
    MOLECULAR_STRUCTURE,    
    VIRAL_PATHOGENS,    
    BACTERIAL_PATHOGENS,    
    OTHER_PATHOGENS,    
    WASTEWATER, 
    WARNING_SYSTEMS,    
    POLICIES, 
    GENERIC_KEYWORDS,   
    COVID_SPECIFIC_KEYWORDS,    
    EPIDEMIC_MEASURES 
    )

In [48]:
words_dictionary = {
'BACTERIAL PATHOGENS' : BACTERIAL_PATHOGENS,
'SEQUENCING METHODS' : SEQUENCING_METHODS,
'METHODS' : METHODS,
'GENOME-BASED ANALYSES' : GENOME_BASED_ANALYSES,
'FIELD': FIELD,
'EVOLUTIONARY MECHANISMS' : EVOLUTIONARY_MECHANISMS,
'TRANSMISSION EPIDEMIOLOGY': TRANSMISSION_EPIDEMIOLOGY,  
'SURVEILLANCE AND PUBLIC HEALTH': SURVEILLANCE_AND_PUBLIC_HEALTH,
'PLATFORM AND TECHNOLOGIES' : PLATFORM_AND_TECHNOLOGIES,
'BIOINFORMATIC TOOLS' : BIOINFORMATIC_TOOLS,
'CONSORTIA AND DATABASES' : CONSORTIA_AND_DATABASES,
'VARIANT AND GENETIC MARKER' : VARIANT_AND_GENETIC_MARKER,
'IMMUNOLOGY TOOLS AND CONCEPTS' : IMMUNOLOGY_TOOLS_AND_CONCEPTS,
'HAI': HAI,
'AMR' : AMR,
'MOLECULAR STRUCTURE' : MOLECULAR_STRUCTURE,
'VIRAL PATHOGENS' : VIRAL_PATHOGENS,
'BACTERIAL PATHOGENS' : BACTERIAL_PATHOGENS,
'OTHER PATHOGENS' : OTHER_PATHOGENS,
'WASTEWATER' : WASTEWATER + ['wastewater'],
'WARNING SYSTEMS' : WARNING_SYSTEMS,
'POLICIES' : POLICIES,
'GENERIC KEYWORDS' : GENERIC_KEYWORDS,
'COVID-SPECIFIC KEYWORDS' : COVID_SPECIFIC_KEYWORDS,
'EPIDEMIC MEASURES' : EPIDEMIC_MEASURES
}

In [49]:
def search_keywords_by_category(df_polars: pl.DataFrame | pl.LazyFrame, words_dictionary : dict):

    new_columns = []

    for category, keywords in words_dictionary.items():

        pattern = '|'.join( [ re.escape(kw).replace(r'\-', r'[-\s]') for kw in keywords] )

        extracted = pl.col('text').str.replace('-',' ').str.extract_all(f'(?i)\\b({pattern})\\b')

        new_columns.extend([

        extracted.alias(f'{category}_matched'),

        ])

    return df_polars.with_columns(new_columns)

In [50]:
def count_list_elements(
    df: pl.DataFrame | pl.LazyFrame
) -> pl.DataFrame | pl.LazyFrame:

    count_expressions = []

    for col_name, dtype in df.collect_schema().items():

        if isinstance(dtype, pl.List) and dtype.inner == pl.Utf8:

            count_expr = (
                (pl.col(col_name).list.len() > 0)
                .cast(pl.Int8)
                .alias(f'n_{col_name}')
            )

            count_expressions.append(count_expr)

    if count_expressions:
        return df.with_columns(count_expressions)

    return df


In [51]:
polars_keywords_matched = search_keywords_by_category(class_eval_df_polars,words_dictionary=words_dictionary)

In [52]:
polars_keywords_matched = count_list_elements(polars_keywords_matched)

In [53]:
polars_keywords_matched = polars_keywords_matched.drop('text')

In [54]:
pandas_keywords_matched = polars_keywords_matched.to_pandas()

In [55]:
cluster_list = list(range(n_clusters))

In [56]:
class_eval_df_analysis = pandas_keywords_matched[ pandas_keywords_matched['Cluster'].isin(cluster_list) ]

In [57]:
if len(cluster_list) > 1:
    # Settare un identificativo per Cluster
    if 'Cluster Label' in class_eval_df_analysis.columns:
        class_eval_df_analysis['Graph_label'] = class_eval_df_analysis['Cluster Label']
    else:
        class_eval_df_analysis['Graph_label'] = class_eval_df_analysis['Cluster']
else:
    # Settare un identificativo per Topic Label
    class_eval_df_analysis['Graph_label'] = class_eval_df_analysis['Topic Label']

In [58]:
result = (
    class_eval_df_analysis
        .groupby('Graph_label')
        .agg(
            **{col: (col, 'mean') for col in class_eval_df_analysis.columns if col.startswith('n_')},
            n_rows=('Graph_label', 'size')
        )
)

In [59]:
heatmap_df = result.drop(columns=['n_rows'])


In [60]:
heatmap_df = heatmap_df.rename(
    columns=lambda c: (
        c.replace('n_', '')
         .replace('_matched', '')
         .replace('_', ' ')
         .title()
    )
)


In [ ]:
import plotly.graph_objects as go
import numpy as np

# Ordina colonne per media decrescente e righe per somma decrescente
ordered_cols = heatmap_df.mean().sort_values(ascending=False).index
ordered_rows = heatmap_df.sum(axis=1).sort_values(ascending=False).index
df_sorted = heatmap_df.loc[ordered_rows, ordered_cols]

n_cols = len(df_sorted.columns)
fig_width = max(900, 45 * n_cols)

# Normalizza i valori per le annotazioni
z = df_sorted.values
z_text = np.round(z, 2).astype(str)

fig = go.Figure(data=go.Heatmap(
    z=z,
    x=list(df_sorted.columns),
    y=list(df_sorted.index),
    text=z_text,
    texttemplate="%{text}",
    textfont={"size": 9},
    colorscale="RdYlGn",    
    colorbar=dict(
        title=dict(text="Value", side="right"),
        thickness=15,
        len=0.8,
        tickfont=dict(size=11),
    ),
    hoverongaps=False,
    hovertemplate=(
        "<b>Row:</b> %{y}<br>"
        "<b>Column:</b> %{x}<br>"
        "<b>Value:</b> %{z:.3f}<extra></extra>"
    ),
))

fig.update_layout(
    title=dict(
        text="Heatmap — Class Evaluation",
        font=dict(size=18, color="#2c3e50"),
        x=0.5,
        xanchor="center"
    ),
    width=fig_width,
    height=max(500, 30 * len(df_sorted)), 
    margin=dict(l=120, r=80, t=80, b=140),
    font=dict(family="Arial, sans-serif", size=11, color="#2c3e50"),
    paper_bgcolor="#f8f9fa",
    plot_bgcolor="#ffffff",
    xaxis=dict(
        tickangle=-40,
        tickfont=dict(size=10),
        side="bottom",
        title="",
    ),
    yaxis=dict(
        tickfont=dict(size=10),
        autorange="reversed", 
        title="",
    ),
)

# Linee di separazione tra celle
fig.update_traces(
    xgap=1,
    ygap=1,
)

fig.show()


fig.write_image(
    "./figures/model_class_evaluation_figures/validation_heatmap.pdf",
)